# San Fernando ITT - Geovisor Interactivo

Notebook para el analisis geoespacial del Corredor San Fernando.

**Flujo de trabajo:**
1. Configuracion del entorno (Colab o local)
2. Carga del shapefile (MAGNA-SIRGAS) y transformacion a WGS84
3. Validacion de calidad de datos
4. Conversion a GeoJSON
5. Generacion del geovisor interactivo con Folium
6. Control de capas y edicion de poligono
7. Exportacion de resultados
8. Filtrado espacial con poligono personalizado
9. Extraccion de POIs de OpenStreetMap (salud, educacion, comercio, etc.)
10. Geovisor con POIs clasificados por categoria
11. Exportacion de POIs (GeoJSON y CSV)
12. Mapa HTML final

## 1. Configuracion del entorno

Se detecta automaticamente si se ejecuta en Google Colab o localmente.
- En Colab: clona el repositorio y usa esa ruta.
- En local: usa rutas relativas desde Notebooks/.

In [5]:
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules

# Instalar dependencias base
if EN_COLAB:
    !pip install -q geopandas folium mapclassify
    # Clonar el repositorio
    !rm -rf Corredor_San_fernando
    !git clone https://github.com/j0rg3c45/Corredor_San_fernando.git
    ruta_base = Path('Corredor_San_fernando')
else:
    # Ejecucion local (desde carpeta Notebooks/)
    ruta_base = Path('..')

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Ruta base: {ruta_base.resolve()}")

Cloning into 'Corredor_San_fernando'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 34 (delta 6), reused 32 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 10.97 KiB | 10.97 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Entorno: Google Colab
Ruta base: /content/Corredor_San_fernando


## 2. Importaciones

In [6]:
import geopandas as gpd
import folium
from folium.plugins import Draw
import json

## 3. Configuracion de rutas

In [7]:
# Rutas del proyecto
ruta_shapefile = ruta_base / 'Data' / 'shape_geo' / 'Corredor_San_fernando.shp'
ruta_geojson = ruta_base / 'Data' / 'Corredor_San_fernando.geojson'
ruta_outputs = ruta_base / 'Outputs'

# Verificar que el shapefile existe
if ruta_shapefile.exists():
    print(f"Shapefile encontrado: {ruta_shapefile}")
else:
    raise FileNotFoundError(
        f"No se encontro el shapefile en: {ruta_shapefile.resolve()}\n"
        f"Verifica que el repositorio se clono correctamente."
    )

Shapefile encontrado: Corredor_San_fernando/Data/shape_geo/Corredor_San_fernando.shp


## 4. Carga y transformacion del shapefile

El shapefile original esta en **MAGNA-SIRGAS** (sistema de referencia colombiano/mexicano).
Se reproyecta a **EPSG:4326 (WGS84)** para garantizar consistencia con el estandar del proyecto.

In [8]:
# Cargar shapefile
gdf_corredor = gpd.read_file(ruta_shapefile)

# Mostrar informacion del CRS original
print(f"CRS original: {gdf_corredor.crs}")
print(f"Registros: {len(gdf_corredor)}")
print(f"Columnas: {list(gdf_corredor.columns)}")
print(f"Tipos de geometria: {gdf_corredor.geom_type.unique().tolist()}")

# Reproyectar a WGS84 (EPSG:4326)
if gdf_corredor.crs is None:
    # Si no tiene CRS definido, asignar MAGNA-SIRGAS origen y luego transformar
    print("\nADVERTENCIA: Sin CRS definido. Asignando MAGNA-SIRGAS (EPSG:4686)...")
    gdf_corredor = gdf_corredor.set_crs(epsg=4686)
    gdf_corredor = gdf_corredor.to_crs(epsg=4326)
    print("Reproyectado a WGS84.")
elif gdf_corredor.crs.to_epsg() != 4326:
    crs_origen = gdf_corredor.crs
    gdf_corredor = gdf_corredor.to_crs(epsg=4326)
    print(f"\nReproyectado de {crs_origen} a EPSG:4326 (WGS84).")
else:
    print("\nEl shapefile ya esta en EPSG:4326 (WGS84).")

print(f"CRS final: {gdf_corredor.crs}")
gdf_corredor.head()

CRS original: EPSG:3115
Registros: 1
Columnas: ['Id', 'geometry']
Tipos de geometria: ['Polygon']

Reproyectado de EPSG:3115 a EPSG:4326 (WGS84).
CRS final: EPSG:4326


,Id,geometry
0,0,"POLYGON ((-76.53468 3.42265, -76.53535 3.42089..."


## 5. Validacion de calidad de datos

Verificar completitud y consistencia segun los lineamientos del proyecto.

In [9]:
# Validacion de geometrias
geometrias_invalidas = gdf_corredor[~gdf_corredor.geometry.is_valid]
geometrias_vacias = gdf_corredor[gdf_corredor.geometry.is_empty]

print("--- Reporte de calidad de datos ---")
print(f"Total de registros: {len(gdf_corredor)}")
print(f"Geometrias invalidas: {len(geometrias_invalidas)}")
print(f"Geometrias vacias: {len(geometrias_vacias)}")
print(f"CRS: {gdf_corredor.crs}")
print(f"Bounds (minx, miny, maxx, maxy): {gdf_corredor.total_bounds}")

# Valores nulos por columna
nulos = gdf_corredor.isnull().sum()
if nulos.any():
    print(f"\nValores nulos por columna:")
    print(nulos[nulos > 0])

# Corregir geometrias invalidas si las hay
if len(geometrias_invalidas) > 0:
    print("\nCorrigiendo geometrias invalidas con buffer(0)...")
    gdf_corredor['geometry'] = gdf_corredor.geometry.buffer(0)
    corregidas = gdf_corredor[~gdf_corredor.geometry.is_valid]
    print(f"Geometrias invalidas despues de correccion: {len(corregidas)}")

--- Reporte de calidad de datos ---
Total de registros: 1
Geometrias invalidas: 0
Geometrias vacias: 0
CRS: EPSG:4326
Bounds (minx, miny, maxx, maxy): [-76.54983987   3.41798405 -76.53317894   3.43406679]


## 6. Exportar a GeoJSON

Conversion a formato GeoJSON (RFC 7946) en WGS84.

In [10]:
# Exportar a GeoJSON
gdf_corredor.to_file(ruta_geojson, driver='GeoJSON')
print(f"GeoJSON exportado en: {ruta_geojson}")
print(f"Tamano: {ruta_geojson.stat().st_size / 1024:.1f} KB")

GeoJSON exportado en: Corredor_San_fernando/Data/Corredor_San_fernando.geojson
Tamano: 2.2 KB


## 7. Geovisor interactivo con Folium

Mapa interactivo con:
- Capa del poligono del Corredor San Fernando
- Control de capas (LayerControl)
- Herramienta de dibujo (Draw) para editar/crear poligonos en el visor
- El poligono dibujado se puede exportar y usar para filtrar datos

In [11]:
# Calcular centroide para centrar el mapa
centroide = gdf_corredor.union_all().centroid
centro = [centroide.y, centroide.x]

# Crear mapa base
mapa = folium.Map(
    location=centro,
    zoom_start=14,
    tiles='OpenStreetMap'
)

# Agregar capa del corredor
columnas_tooltip = gdf_corredor.columns.drop('geometry').tolist()[:3]

capa_corredor = folium.GeoJson(
    gdf_corredor.to_json(),
    name='Corredor San Fernando',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#3388ff',
        'weight': 2,
        'fillOpacity': 0.3
    },
    tooltip=folium.GeoJsonTooltip(
        fields=columnas_tooltip,
        aliases=columnas_tooltip
    ) if columnas_tooltip else None
)
capa_corredor.add_to(mapa)

# Capas base adicionales
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(mapa)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Oscuro').add_to(mapa)

# Herramienta de dibujo para editar poligonos en el visor
draw = Draw(
    export=True,
    draw_options={
        'polyline': False,
        'rectangle': True,
        'polygon': True,
        'circle': False,
        'marker': False,
        'circlemarker': False
    },
    edit_options={
        'edit': True,
        'remove': True
    }
)
draw.add_to(mapa)

# Control de capas
folium.LayerControl(collapsed=False).add_to(mapa)

# Mostrar mapa usando IFrame para evitar el mensaje 'Trust Notebook'
from IPython.display import IFrame, display
import tempfile, os

if EN_COLAB:
    # En Colab se puede mostrar directamente sin problema
    display(mapa)
else:
    # En Jupyter local / VS Code, guardar como HTML temporal y mostrar con IFrame
    ruta_outputs.mkdir(parents=True, exist_ok=True)
    ruta_temp_mapa = ruta_outputs / 'geovisor_corredor_san_fernando.html'
    mapa.save(str(ruta_temp_mapa))
    display(IFrame(src=str(ruta_temp_mapa), width='100%', height=600))

## 8. Filtrado espacial con poligono personalizado

Despues de dibujar un poligono en el mapa anterior:
1. Haz clic en "Export" (boton en el mapa)
2. Guarda el archivo como `Data/filtro_poligono.geojson`
3. Ejecuta la siguiente celda para filtrar los datos

In [12]:
# Filtrado espacial con poligono dibujado por el usuario
ruta_filtro = ruta_base / 'Data' / 'filtro_poligono.geojson'

if ruta_filtro.exists():
    gdf_filtro = gpd.read_file(ruta_filtro).to_crs(epsg=4326)

    # Filtrar datos que intersectan con el poligono dibujado
    datos_filtrados = gpd.overlay(gdf_corredor, gdf_filtro, how='intersection')

    print(f"Registros originales: {len(gdf_corredor)}")
    print(f"Registros dentro del poligono de filtro: {len(datos_filtrados)}")
    datos_filtrados.head()
else:
    print("No se encontro archivo de filtro en Data/filtro_poligono.geojson")
    print("Dibuja un poligono en el mapa, exportalo y guardalo en esa ruta.")

No se encontro archivo de filtro en Data/filtro_poligono.geojson
Dibuja un poligono en el mapa, exportalo y guardalo en esa ruta.


## 9. Extraccion de Puntos de Interes (POIs) desde OpenStreetMap

Se utiliza `osmnx` para extraer todos los puntos de interes dentro del poligono del Corredor San Fernando.

Categorias extraidas:
- **Salud:** hospitales, clinicas, farmacias, consultorios
- **Educacion:** escuelas, universidades, jardines infantiles
- **Comercio:** tiendas, supermercados, restaurantes
- **Servicios:** bancos, cajeros, oficinas gubernamentales
- **Transporte:** paradas de bus, estaciones

Fuente: OpenStreetMap (datos abiertos, licencia ODbL).

In [ ]:
# Instalar osmnx (tiene dependencias pesadas, puede tardar ~1 min)
if EN_COLAB:
    !pip install -q osmnx

import osmnx as ox

print(f"osmnx version: {ox.__version__}")

In [ ]:
# Definir tags de interes por categoria
# Referencia: https://wiki.openstreetmap.org/wiki/Map_features
tags_pois = {
    "amenity": True,       # Salud, educacion, servicios publicos, restaurantes
    "shop": True,          # Comercios de todo tipo
    "healthcare": True,    # Servicios de salud especificos
    "public_transport": True,  # Paradas y estaciones
    "office": True,        # Oficinas (gobierno, empresas)
    "tourism": True        # Sitios turisticos, hoteles
}

# Obtener el poligono de busqueda
poligono_busqueda = gdf_corredor.union_all()

print(f"Area de busqueda: {poligono_busqueda.area:.6f} grados cuadrados")
print(f"Bounds: {poligono_busqueda.bounds}")
print("\nExtrayendo POIs de OpenStreetMap...")

In [ ]:
# Extraer POIs dentro del poligono del corredor
gdf_pois = ox.features_from_polygon(poligono_busqueda, tags=tags_pois)

print(f"POIs encontrados: {len(gdf_pois)}")
print(f"\nColumnas disponibles: {len(gdf_pois.columns)}")
print(f"Tipos de geometria: {gdf_pois.geom_type.value_counts().to_dict()}")

In [ ]:
# Convertir todas las geometrias a puntos (centroide) para visualizacion uniforme
gdf_pois_puntos = gdf_pois.copy()
gdf_pois_puntos['geometry'] = gdf_pois_puntos.geometry.centroid

# Asegurar CRS WGS84
if gdf_pois_puntos.crs is None:
    gdf_pois_puntos = gdf_pois_puntos.set_crs(epsg=4326)
elif gdf_pois_puntos.crs.to_epsg() != 4326:
    gdf_pois_puntos = gdf_pois_puntos.to_crs(epsg=4326)

# Crear columna de categoria unificada
def clasificar_poi(row):
    """Clasifica un POI en categoria principal."""
    if pd.notna(row.get('healthcare')):
        return 'Salud'
    amenity = row.get('amenity', '')
    if amenity in ['hospital', 'clinic', 'pharmacy', 'doctors', 'dentist']:
        return 'Salud'
    elif amenity in ['school', 'university', 'kindergarten', 'college', 'library']:
        return 'Educacion'
    elif amenity in ['restaurant', 'cafe', 'fast_food', 'bar', 'pub']:
        return 'Gastronomia'
    elif amenity in ['bank', 'atm', 'post_office', 'police', 'fire_station']:
        return 'Servicios'
    elif amenity in ['bus_station', 'fuel', 'parking']:
        return 'Transporte'
    elif pd.notna(row.get('shop')):
        return 'Comercio'
    elif pd.notna(row.get('public_transport')):
        return 'Transporte'
    elif pd.notna(row.get('tourism')):
        return 'Turismo'
    elif pd.notna(row.get('office')):
        return 'Oficinas'
    else:
        return 'Otro'

import pandas as pd
gdf_pois_puntos['categoria'] = gdf_pois_puntos.apply(clasificar_poi, axis=1)

# Seleccionar columnas relevantes
columnas_interes = ['name', 'amenity', 'shop', 'healthcare', 'categoria', 'geometry']
columnas_disponibles = [c for c in columnas_interes if c in gdf_pois_puntos.columns]
gdf_pois_limpio = gdf_pois_puntos[columnas_disponibles].copy()

print("\n--- Resumen de POIs por categoria ---")
print(gdf_pois_limpio['categoria'].value_counts())
print(f"\nTotal POIs con nombre: {gdf_pois_limpio['name'].notna().sum()}")
gdf_pois_limpio.head(10)

## 10. Geovisor con POIs extraidos

Mapa interactivo que muestra:
- Poligono del corredor
- POIs clasificados por categoria con colores diferenciados
- Control de capas para encender/apagar categorias

In [ ]:
# Colores por categoria
colores_categoria = {
    'Salud': 'red',
    'Educacion': 'blue',
    'Comercio': 'green',
    'Gastronomia': 'orange',
    'Servicios': 'purple',
    'Transporte': 'darkblue',
    'Turismo': 'cadetblue',
    'Oficinas': 'gray',
    'Otro': 'lightgray'
}

# Crear mapa con POIs
mapa_pois = folium.Map(
    location=centro,
    zoom_start=15,
    tiles='CartoDB positron'
)

# Agregar poligono del corredor
folium.GeoJson(
    gdf_corredor.to_json(),
    name='Corredor San Fernando',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#3388ff',
        'weight': 2,
        'fillOpacity': 0.1
    }
).add_to(mapa_pois)

# Agregar POIs por categoria como FeatureGroups (para control de capas)
for categoria, color in colores_categoria.items():
    pois_cat = gdf_pois_limpio[gdf_pois_limpio['categoria'] == categoria]
    if len(pois_cat) == 0:
        continue

    grupo = folium.FeatureGroup(name=f"{categoria} ({len(pois_cat)})")

    for idx, row in pois_cat.iterrows():
        nombre = row.get('name', 'Sin nombre')
        if pd.isna(nombre):
            nombre = 'Sin nombre'

        popup_text = f"<b>{nombre}</b><br>Categoria: {categoria}"
        if pd.notna(row.get('amenity')):
            popup_text += f"<br>Amenity: {row['amenity']}"
        if pd.notna(row.get('shop')):
            popup_text += f"<br>Shop: {row['shop']}"

        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=6,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_text, max_width=250),
            tooltip=nombre
        ).add_to(grupo)

    grupo.add_to(mapa_pois)

# Control de capas
folium.LayerControl(collapsed=False).add_to(mapa_pois)

# Mostrar mapa
from IPython.display import IFrame, display

if EN_COLAB:
    display(mapa_pois)
else:
    ruta_outputs.mkdir(parents=True, exist_ok=True)
    ruta_temp = ruta_outputs / 'geovisor_pois.html'
    mapa_pois.save(str(ruta_temp))
    display(IFrame(src=str(ruta_temp), width='100%', height=600))

## 11. Exportar POIs a GeoJSON y CSV

Se guardan los POIs extraidos para uso posterior y filtrado.

In [ ]:
# Crear directorio de salida
ruta_outputs.mkdir(parents=True, exist_ok=True)

# Exportar POIs como GeoJSON
ruta_pois_geojson = ruta_outputs / 'pois_corredor_san_fernando.geojson'
gdf_pois_limpio.to_file(ruta_pois_geojson, driver='GeoJSON')
print(f"POIs exportados en GeoJSON: {ruta_pois_geojson}")

# Exportar tambien como CSV (sin geometria, con lat/lon)
df_pois_csv = gdf_pois_limpio.copy()
df_pois_csv['latitud'] = gdf_pois_limpio.geometry.y
df_pois_csv['longitud'] = gdf_pois_limpio.geometry.x
df_pois_csv = df_pois_csv.drop(columns=['geometry'])

ruta_pois_csv = ruta_outputs / 'pois_corredor_san_fernando.csv'
df_pois_csv.to_csv(ruta_pois_csv, index=False, encoding='utf-8')
print(f"POIs exportados en CSV: {ruta_pois_csv}")
print(f"\nTotal registros exportados: {len(df_pois_csv)}")

## 12. Exportar mapa HTML final

Se guarda el geovisor completo (con POIs) como archivo HTML interactivo en Outputs/.

In [ ]:
# Crear directorio de salida si no existe
ruta_outputs.mkdir(parents=True, exist_ok=True)

# Exportar mapa como HTML
ruta_mapa_html = ruta_outputs / 'geovisor_corredor_san_fernando.html'
mapa_pois.save(str(ruta_mapa_html))
print(f"Mapa exportado en: {ruta_mapa_html}")